In [1]:
pip install streamlit pandas numpy scikit-learn plotly folium streamlit-folium

   ---------------------------------------- 0.0/536.9 kB ? eta -:--:--
   ---------------------------------------  524.3/536.9 kB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 536.9/536.9 kB 2.7 MB/s  0:00:00

   ------------- -------------------------- 1/3 [folium]
   ------------- -------------------------- 1/3 [folium]
   ---------------------------------------- 3/3 [streamlit-folium]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
import joblib

# 1. توليد بيانات تدريب موسعة ومعايرة بناءً على مؤشرات الأراضي الجافة
np.random.seed(42)
n_samples = 3000

temp = np.random.uniform(18.0, 49.0, n_samples)
rainfall = np.random.uniform(5.0, 450.0, n_samples)
moisture = np.random.uniform(4.0, 75.0, n_samples)
ndvi = np.random.uniform(0.04, 0.85, n_samples)

# معادلة حساب مؤشر المخاطر البيئية (Calibrated Risk Index)
risk_score = (temp * 0.38) - (rainfall * 0.055) - (moisture * 0.32) - (ndvi * 48.0)

labels = []
for score in risk_score:
    if score > -5.0:
        labels.append(2)  # High Aridity / Severe Risk
    elif score > -23.0:
        labels.append(1)  # Moderate Risk
    else:
        labels.append(0)  # Low Risk / Resilient

df = pd.DataFrame({
    'Temperature': temp,
    'Rainfall': rainfall,
    'Soil_Moisture': moisture,
    'NDVI': ndvi,
    'Risk': labels
})

X = df[['Temperature', 'Rainfall', 'Soil_Moisture', 'NDVI']]
y = df['Risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. بناء نموذج Random Forest محسّن
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=4,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train, y_train)

# 3. استخراج المقاييس للـ Presentation
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print("="*50)
print(f" Model Accuracy: {acc*100:.2f}%")
print(f" Model Weighted F1-Score: {f1*100:.2f}%")
print("="*50)
print("Classification Report:\n", classification_report(y_test, y_pred))

# 4. حفظ المودل المحدث
joblib.dump(model, 'desert_risk_model.pkl')
print(" Successfully exported updated model to 'desert_risk_model.pkl'")

 Model Accuracy: 95.83%
 Model Weighted F1-Score: 95.78%
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.98      0.98       447
           1       0.91      0.91      0.91       140
           2       0.90      0.69      0.78        13

    accuracy                           0.96       600
   macro avg       0.93      0.86      0.89       600
weighted avg       0.96      0.96      0.96       600

 Successfully exported updated model to 'desert_risk_model.pkl'
